## 05-3 트리의 앙상블

**랜덤 포레스트(Random Forest)** 라는 알고리즘에 대해서 알아보자.
---

### 정형 데이터와 비정형 데이터

- 정형 데이터란 쉽게 말해 어떤 구조로 되어 있다는 것을 의미한다.
    - 이런 데이터는 CSV, 데이터베이스 혹은 엑셀에 저장하기가 쉽다
    
&rArr; 지금까지 배운 머신러닝 알고리즘은 정형 데이터에 잘 맞는다.\
(그중 정형 데이터를 다루는 데 가장 뛰어난 성과를 내는 알고리즘이 **앙상블 학습(ensemble learning)** 이다.)

- 이와 반대되는 비정형 데이터는 앞서 말한 방식으로 저장하기 어려운 데이터들을 말한다.
    - 보통 사진, 음악 등이 해당된다.

&rArr; 비정형 데이터에는 나중에 배울 신경망 알고리즘을 적용하게 된다.

---

### 랜덤 포레스트

랜덤 포레스트(Random Forest)는 앙상블 학습의 대표적인 것 중 하나로,\
 안정적인 성능 덕분에 앙상블 학습을 적용할 때 가장 먼저 고려하게 되기도 한다.

#### 동작 방식
1. 부트스트램 샘플 생성(Bootstarp Sampling)
    - 원본 데이터에서 **중복 허용 랜덤 샘플링**
    - 이렇게 만든 데이터 &rArr; **부투스트랩 샘플**

2. 여러 개의 결정 트리 학습
    - 각 트리는 **서로 다른 데이터(부트스트랩 샘플)** 로 학습된다.
    - 추가로 :
            - 각 노드에서 일부 feature만 랜덤 선택

3. 예측
    - 분류 문제 : 다수결 (Voting)
    - 회귀 문제 : 평균 (Average)


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

wine = pd.read_csv("./data/wine_csv_data.csv")
data = wine[["alcohol", "sugar", "pH"]]
target = wine["class"]
train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42
)

In [3]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(rf, train_input, train_target,
                        return_train_score=True, n_jobs=-1)
print(np.mean(scores["train_score"]), np.mean(scores["test_score"]))

0.9973541965122431 0.8905151032797809


출력된 결과를 확인하면 과대적합된 것을 확인할 수 있지만,\
알고리즘을 알아보는 것이 목적이기 때문에 매개변수를 더 조정하지 않도록 한다.

<br>

랜덤 포레스트는 결정 트리의 앙상블이기 때문에 `DecisionTreeClassifier`가 제공하는 중요 매개변수를 모두 제공한다.
- `criterion`, `max_depth`, `max_features`, `min_samples_split`, `min_impurity_decrease`, `min_samples_leaf` 등

결정 트리의 장점 중 하나는 특성 중요도를 계산한다는 것이다.\
랜덤 포레스트의 중요도는 각 결정 트리의 특성 중요도를 취합한 결과이다.

In [4]:
rf.fit(train_input, train_target)

print(rf.feature_importances_)

[0.23167441 0.50039841 0.26792718]


앞서 학습시켰던 결정 트리의 중요도 결과(`[0.1234, 0.8686, 0.0079]`)와 비교하면,  
두 번째 특성인 **당도**의 중요도는 감소하고,  
**도수**와 **pH** 특성의 중요도는 상대적으로 상승했다.

<br>

이러한 변화는 랜덤 포레스트의 학습 방식에서 비롯된다.

- 랜덤 포레스트는 각 트리를 만들 때  
  → **일부 특성을 랜덤하게 선택하여 학습**한다.

- 따라서 특정 하나의 특성(예: 당도)에만 의존하지 않고  
  → 다양한 특성이 학습에 참여하게 된다.

<br>

즉,
> 하나의 특성에 과도하게 집중되는 현상이 완화되고,  
> 여러 특성이 고르게 기여할 기회를 얻게 된다.

<br>
이로 인해 다음과 같은 효과가 나타난다:

- 과대적합(Overfitting) 감소  
- 모델의 일반화 성능(Generalization) 향상  

<br>

---

랜덤 포레스트는 훈련 과정에서  
→ **부트스트랩 샘플(중복 허용 샘플링)** 을 사용한다.

이 과정에서:

- 일부 샘플은 여러 번 선택되기도 하고  
- 일부 샘플은 **아예 선택되지 않을 수도 있다**

이렇게 선택되지 않고 남은 샘플을  
→ **OOB (Out-Of-Bag)** 라고 한다.

<br>

👉 OOB의 역할:

- 각 트리를 학습할 때 사용되지 않은 데이터이기 때문에  
  → **검증 데이터처럼 활용 가능**

- 별도의 검증 세트를 만들지 않아도  
  → **모델 성능을 평가할 수 있음**

In [5]:
# OOB점수 출력을 위해서는 ooob_score 매개변수에 True 를 지정해야 한다.
rf = RandomForestClassifier(oob_score=True, n_jobs=-1, random_state=42)
rf.fit(train_input, train_target)
print(rf.oob_score_)

0.8934000384837406


---

### 엑스트라 트리

엑스트라 트리(Extra Trees)는 랜덤 포레스트와 유사한 앙상블 모델이지만,  
**더 강한 랜덤성(randomness)** 을 적용한 모델이다.

엑스트라 트리는 다음과 같은 차이를 가진다:

- **부트스트랩 샘플을 사용하지 않음**
  - 랜덤 포레스트: 중복 허용 샘플링 (bootstrap)
  - 엑스트라 트리: **전체 훈련 데이터를 그대로 사용**

즉,

> 모든 트리가 동일한 데이터로 학습되지만,  
> 다른 방식으로 분할되어 다양성을 확보한다.


- 노드 분할 방식에 있어서도 차이를 보인다.
    - 랜덤 포레스트:
        - 여러 후보를 비교하여  
        → **가장 좋은 분할 기준을 선택**

    - 엑스트라 트리:
        - 분할 기준을 평가하지 않고  
        → **무작위로 선택**

<br>

예시:

- 랜덤 포레스트:
  - `당도 > 4.3`, `당도 > 5.1`, `당도 > 6.0` 등 비교
  - → 가장 성능 좋은 기준 선택

- 엑스트라 트리:
  - 그냥 랜덤으로 하나 선택
  - → 예: `당도 > 5.1`

<br>

즉,

> 최적의 분할을 찾지 않고,  
> 랜덤한 기준으로 데이터를 나눈다.

In [7]:
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(et, train_input, train_target,
                        return_train_score=True, n_jobs=-1)

print(np.mean(scores["train_score"]), np.mean(scores["test_score"]))

et.fit(train_input, train_target)   
print(et.feature_importances_)

0.9974503966084433 0.8887848893166506
[0.20183568 0.52242907 0.27573525]


현재 예제에서는 특성이 많지 않아 두 모델의 차이가 크지 않았다.

보통 엑스트라 트리가 무작위성이 더 크기 때문에 더 많은 결정 트리를 훈련시켜야 한다.\
하지만, 랜덤하게 노드를 분할하기 때문에 빠른 계산 속도가 장점이 된다.

---

### 그레디언트 부스팅

그레디언트 부스팅은 여러 개의 **얕은 결정 트리(weak learner)** 를 순차적으로 학습시키면서  
이전 모델의 **오차를 점점 줄여나가는 앙상블 방법**이다.

> 얕은 트리를 사용하기 때문에, 노이즈까지 다 외우는 상황이 일어나지 않게되고\
> 그에 따라 과대적합이 발생 확률이 적다.\
> 즉, 
>  - 디테일 못 잡음 &rArr; bias는 크다.
>  - 대신 노이즈는 안 외움 &rArr; variance 낮음

<br>

동작 방식은 다음과 같다 :
1. 초기 모델로 예측 수행  
2. 실제값과의 차이(오차) 계산  
3. 이 오차를 줄이도록 다음 트리 학습  
4. 이 과정을 반복하며 성능 개선


In [8]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
scores = cross_validate(gb, train_input, train_target,
                        return_train_score=True, n_jobs=-1)

print(np.mean(scores["train_score"]), np.mean(scores["test_score"]))

0.8881086892152563 0.8720430147331015


In [9]:
# 결정 트리의 개수를 늘리고 학습률을 증가시키면 성능이 향상될 수 있으며,
# 그에 따라 과대적합도 잘 발생하지 않는다.
gb = GradientBoostingClassifier(n_estimators=500, learning_rate=0.2,
                                random_state=42)
scores = cross_validate(gb, train_input, train_target,
                        return_train_score=True, n_jobs=-1)

print(np.mean(scores["train_score"]), np.mean(scores["test_score"]))

0.9464595437171814 0.8780082549788999


트리 훈련에 사용할 훈련 세트의 비율을 정하는 `subsample`이 존재한다. (기본값은 1)
- 해당 값을 1로 하면 전체 훈련 세트를 사용한다.
- 단, 1보다 작게 하는 경우에는,
    - 훈련 세트의 일부를 사용한다.

&rArr; 이는 마치 경사 하강법에서 일부 샘플을 랜덤으로 선택하는 "확률적 경사 하강법"이나 "미니배치 경사 하강법"과 유사하다.

---

### 히스토그램 기반 그레이디언트 부스팅

히스토그램 기반 그레이디언트 부스팅(Histogram-based Gradient Boosting)은 정형 데이터를 다루는 알고리즘 중에 가장 인기가 많다.

연속형 데이터를 그대로 쓰지 않고,\
구간(bin)으로 나누어 히스토그램을 변환하는 것이 핵심 아이디어이다.

동작 방식은 다음과 같다 :

1. 각 feature 값을 일정한 구간(bin)으로 나눔  
2. 각 구간별로 통계값(gradient, hessian 등) 계산  
3. 이 정보를 바탕으로 최적 분할 수행  

In [12]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(random_state=42)
scores = cross_validate(hgb, train_input, train_target, 
                        return_train_score=True)

print(np.mean(scores["train_score"]), np.mean(scores["test_score"]))

0.9321723946453317 0.8801241948619236


In [15]:
from sklearn.inspection import permutation_importance

hgb.fit(train_input, train_target)
result = permutation_importance(hgb, test_input, test_target,
                                n_repeats=10, random_state=42, n_jobs=-1)

print(result.importances_mean)

hgb.score(test_input, test_target)

[0.05969231 0.20238462 0.049     ]


0.8723076923076923

사이킷런에서 제공하고 있는 히스토그램 기반 그레이디언트 부스팅 말고도,\
이를 구현한 라이브러리가 여럿 존재한다.

&rArr; 가장 대표적인 것이 바로 `XGBoost`와 `LightGBM`이다.

---

#### 모델별 핵심 정리

1) 결정 트리
- 특징: 
    - 데이터를 계속 쪼개면서 학습
    - 비선형 패턴을 잘 잡는다.
- 문제 : 너무 잘 맞춰서 과적합 위험성 존재

2) 랜덤 포레스트
- 구조 : 
    - 부트스트랩 샘플
    - feature 랜덤 선택
    - 여러 트리 평균
- 효과 : 과적합 완화

3) 엑스트라 트리
- 랜덤 포레스트보다 더 랜덤
    - 부트스트랩이 아닌 전체 데이터 사용
    - 분할 기준도 랜덤
- 효과 : 다양성 증가, 속도 증가, 과적합을 더 완화시킬 수 있다.

4) 그레이디언트 부스팅
- 구조 :
    - 트리를 순차적으로 학습
    - 이전 모델의 오차를 보정
- 특징 :
    - 얕은 트리를 사용 + learning_rate 존재
- 효과 : bias가 감소하고, 높은 성능을 뽑을 수 있다.

5) 히스토그램 기반 GB
- 핵심 개선 : 데이터를 bin(구간)으로 나눔
- 효과 : 속도 증가, 메모리 감소, 대용량 데이터에 강하다.

#### XGBoost & LightGBM
- XGBoost = 최적화된 Gradient Boosting
    - 특징 : 
        - 정규화 포함(과적합 방지)
        - 결측값 자동 처리
        - 높은 성능
    - 용도로는 회귀 / 분류 / 랭킹 모두 가능하다.

- LightGBM = 히스토그램 기반 GB의 고속 최적화 버전
    - 특징 :
        - Histogram 기반
        - Leaf-wise 성장
            - 기존 트리는 level-wise(균형있게 성장)
            - LightGBM은 **leaf-wise(가장 손실 감소 큰 쪽만 계속 확장)**  
            &rArr; 결과적으로 더 빠르게 loss 감소, 더 높은 성능이 가능하다.
    - 학습 속도가 매우 빠르고, 성능도 상당히 좋다. 또한 대규모 데이터에도 최적화 되어있다.
    - 그러나 leaf-wise로 인해 과적합 위험이 존재하며, 작은 데이터에서는 불안정할 수 있다.